In [1]:
import polars as pl
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [2]:
import torch

torch.set_float32_matmul_precision("high")


from gluonts.dataset.pandas import PandasDataset
from gluonts.dataset.split import split
from gluonts.torch import DeepAREstimator

from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger, CSVLogger

import warnings
import logging

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=r".*isinstance\(treespec, LeafSpec\).*")

In [3]:
df = pl.read_csv("../data/01_raw/Walmart_Sales.csv")

In [4]:
df = df.with_columns(pl.col("Date").str.to_date("%d-%m-%Y")).sort("Date")

df

Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
i64,date,f64,i64,f64,f64,f64,f64
1,2010-02-05,1643690.9,0,42.31,2.572,211.096358,8.106
2,2010-02-05,2.1370e6,0,40.19,2.572,210.752605,8.324
3,2010-02-05,461622.22,0,45.71,2.572,214.424881,7.368
4,2010-02-05,2.1351e6,0,43.76,2.598,126.442065,8.623
5,2010-02-05,317173.1,0,39.7,2.572,211.653972,6.566
…,…,…,…,…,…,…,…
41,2012-10-26,1.3165e6,0,41.8,3.686,199.219532,6.195
42,2012-10-26,514756.08,0,70.5,4.301,131.193097,6.943
43,2012-10-26,587603.55,0,69.17,3.506,214.741539,8.839


In [5]:
df = df.with_columns(
    [
        pl.col("Holiday_Flag").cast(pl.Float32),
        pl.col("Temperature").cast(pl.Float32),
        pl.col("Fuel_Price").cast(pl.Float32),
        pl.col("CPI").cast(pl.Float32),
        pl.col("Unemployment").cast(pl.Float32),
    ]
)

df

Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
i64,date,f64,f32,f32,f32,f32,f32
1,2010-02-05,1643690.9,0.0,42.310001,2.572,211.096359,8.106
2,2010-02-05,2.1370e6,0.0,40.189999,2.572,210.752609,8.324
3,2010-02-05,461622.22,0.0,45.709999,2.572,214.424881,7.368
4,2010-02-05,2.1351e6,0.0,43.759998,2.598,126.442062,8.623
5,2010-02-05,317173.1,0.0,39.700001,2.572,211.653976,6.566
…,…,…,…,…,…,…,…
41,2012-10-26,1.3165e6,0.0,41.799999,3.686,199.219528,6.195
42,2012-10-26,514756.08,0.0,70.5,4.301,131.1931,6.943
43,2012-10-26,587603.55,0.0,69.169998,3.506,214.741547,8.839


In [6]:
df_plot = df.with_columns(pl.col("Store").cast(pl.Utf8))
fig = px.line(
    df_plot,
    x="Date",
    y="Weekly_Sales",
    color="Store",
    hover_data={"Date": "|%B %d, %Y"},
    title="Weekly Sales by Store",
)
fig.update_xaxes(dtick="M1", tickformat="%b\n%Y")
fig.show()

In [7]:
df.to_pandas()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,2010-02-05,1643690.90,0.0,42.310001,2.572,211.096359,8.106
1,2,2010-02-05,2136989.46,0.0,40.189999,2.572,210.752609,8.324
2,3,2010-02-05,461622.22,0.0,45.709999,2.572,214.424881,7.368
3,4,2010-02-05,2135143.87,0.0,43.759998,2.598,126.442062,8.623
4,5,2010-02-05,317173.10,0.0,39.700001,2.572,211.653976,6.566
...,...,...,...,...,...,...,...,...
6430,41,2012-10-26,1316542.59,0.0,41.799999,3.686,199.219528,6.195
6431,42,2012-10-26,514756.08,0.0,70.500000,4.301,131.193100,6.943
6432,43,2012-10-26,587603.55,0.0,69.169998,3.506,214.741547,8.839
6433,44,2012-10-26,361067.07,0.0,46.970001,3.755,131.193100,5.217


In [8]:
from lightning.pytorch.callbacks import ModelCheckpoint

stores = df["Store"].unique().sort().to_list()
num_stores = int(df["Store"].max() + 1)

static_features_df = pd.DataFrame({"Store_Cat": stores}, index=stores)

static_features_df["Store_Cat"] = static_features_df["Store_Cat"].astype("category")

dataset = PandasDataset.from_long_dataframe(
    df.to_pandas(),
    target="Weekly_Sales",
    freq="W",
    timestamp="Date",
    item_id="Store",
    feat_dynamic_real=["Holiday_Flag"],
    static_features=static_features_df,
)

# Split the data for training and testing
training_data, test_gen = split(dataset, offset=-36)
test_data = test_gen.generate_instances(prediction_length=12, windows=3)


tb_logger = TensorBoardLogger(save_dir="lightning_logs", name="deepar/tensorboard")

csv_logger = CSVLogger(save_dir="lightning_logs", name="deepar/csv")


checkpoint_callback = ModelCheckpoint(
    filename="deepar-{epoch:02d}",
    save_last=True,
    save_top_k=1,
    every_n_epochs=1,
)


early_stopping_callback = EarlyStopping(
    monitor="train_loss", patience=10, min_delta=0.01, mode="min"
)

# Train the model and make predictions
model = DeepAREstimator(
    prediction_length=12,
    freq="W",
    num_feat_static_cat=1,
    num_feat_dynamic_real=1,
    cardinality=[num_stores],
    batch_size=256,
    num_batches_per_epoch=128,
    num_layers=2,
    hidden_size=128,
    lr=1e-4,
    lags_seq=[1, 2, 4, 13, 52],
    trainer_kwargs={
        "max_epochs": 5,
        "logger":[csv_logger, tb_logger],
                "callbacks": [checkpoint_callback, early_stopping_callback],
        "enable_model_summary": True,
        "enable_progress_bar": False,
    },
).train(training_data)

forecasts = list(model.predict(test_data.input))

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type        | Params | Mode  | In sizes                                                   | Out sizes   
--------------------------------------------------------------------------------------------------------------------------
0 | model | DeepARModel | 217 K  | train | [[1, 1], [1, 1], [1, 63, 4], [1, 63], [1, 63], [1, 12, 4]] | [1, 100, 12]
--------------------------------------------------------------------------------------------------------------------------
217 K     Trainable params
0         Non-trainable params
217 K     Total params
0.870     Total estimated model params size (MB)
11        Modules in train mode
0         Modules in eval mode
Epoch 0, global step 128: 'train_loss' reached 13.69708 (best 13.69708), saving model to 'lightning_logs/deepar/csv/version_0/checkpoints/epoch=0-step=128.ckpt' as top 

In [9]:
df_store1 = df.filter(pl.col("Store") == 1)

forecasts_store1 = [f for f in forecasts if str(f.item_id) == "1"]

fig = go.Figure()


fig.add_trace(
    go.Scatter(
        x=df_store1["Date"],
        y=df_store1["Weekly_Sales"],
        name="Actual Values (Store 1)",
        line=dict(color="#0f172a", width=2),
    )
)


if forecasts_store1:
    forecast_1 = forecasts_store1[0]

    forecast_dates = forecast_1.index.to_timestamp()

    p10 = forecast_1.quantile(0.1)
    p90 = forecast_1.quantile(0.9)
    median = forecast_1.quantile(0.5)

    fig.add_trace(
        go.Scatter(
            x=forecast_dates,
            y=p90,
            mode="lines",
            line=dict(width=0),
            showlegend=False,
            hoverinfo="skip",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=forecast_dates,
            y=p10,
            mode="lines",
            line=dict(width=0),
            fill="tonexty",
            fillcolor="rgba(59, 130, 246, 0.15)",
            name="Confidence Interval (80%)",
            hoverinfo="skip",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=forecast_dates,
            y=median,
            mode="lines+markers",
            line=dict(color="#2563eb", width=2.5, dash="dash"),
            marker=dict(size=4),
            name="Prediction (Median)",
        )
    )
else:
    print("No prediction found for Store 1 in the forecasts set.")


fig.update_layout(
    title=dict(
        text="Actual Sales vs DeepAR Prediction — Store 1",
        font=dict(size=18, color="#0f172a", weight="bold"),
    ),
    xaxis_title="Date",
    yaxis_title="Weekly Sales ($)",
    plot_bgcolor="white",
    paper_bgcolor="white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)

fig.update_xaxes(gridcolor="#f1f5f9", linecolor="#cbd5e1")
fig.update_yaxes(gridcolor="#f1f5f9", linecolor="#cbd5e1", tickprefix="$")

fig.show()